# 🎙️ Interview Feature — Statistical Analysis for Evaluation Design
This notebook analyses the raw acoustic and linguistic features extracted from 86 interviews.  
The goal is to **calibrate an evaluation rubric** — using descriptive statistics, distribution fitting,  
z-score normalization and percentile bands to determine meaningful thresholds for each dimension.

In [6]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import norm, shapiro, kstest, skew, kurtosis
from pathlib import Path
import math

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 60)

OUT_DIR = Path('/Users/samarthsrivastava/Audio Analysis/analysis_output')
OUT_DIR.mkdir(exist_ok=True)
DATA_PATH = '/Users/samarthsrivastava/Audio Analysis/test_extracted_features.jsonl'
print('✅ Ready')

✅ Ready


---
## Section 1 — Data Loading & Flattening

In [7]:
records = []
with open(DATA_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        if obj.get('status') == 'success':
            records.append(obj)

print(f'Loaded {len(records)} successful records')

def safe_get(d, *keys, default=np.nan):
    """Safely traverse nested dict keys."""
    for k in keys:
        if isinstance(d, dict) and k in d:
            d = d[k]
        else:
            return default
    return d if d is not None else default

rows = []
for rec in records:
    f = rec.get('features', {})
    flu = f.get('fluency', {})
    scr = flu.get('scripted_detection', {})
    intel = f.get('intelligibility', {})
    disc = f.get('discourse', {})
    sent = f.get('sentiment', {})
    voice = f.get('voice_modulation', {})
    lex = f.get('lexical_resource', {})

    row = {
        'response_id':              rec.get('response_id', np.nan),
        'duration_s':               rec['duration_ms'] / 1000 if rec.get('duration_ms') is not None else np.nan,
        'fluency_wpm':              safe_get(flu, 'wpm'),
        'fluency_pause_freq':       safe_get(flu, 'pause_frequency'),
        'fluency_mean_pause_dur':   safe_get(flu, 'mean_pause_duration'),
        'fluency_total_words':      safe_get(flu, 'total_words'),
        'fluency_filler_count':     safe_get(flu, 'filler_count'),
        'fluency_filler_rate':      safe_get(flu, 'filler_rate'),
        'fluency_scripted_score':   safe_get(scr, 'ai_scripted_speech_score'),
        'intelligibility_conf':     safe_get(intel, 'mean_confidence'),
        'intelligibility_var_conf': safe_get(intel, 'variance_confidence'),
        'pronounciation_score':     safe_get(intel, 'pronunciation_score'),
        'discourse_connectors':     safe_get(disc, 'connector_count'),
        'discourse_tier1':          safe_get(disc, 'tier1_count'),
        'discourse_tier2':          safe_get(disc, 'tier2_count'),
        'sentiment_compound':       safe_get(sent, 'mean_compound'),
        'sentiment_std':            safe_get(sent, 'std_compound'),
        'sentiment_neg_ratio':      safe_get(sent, 'neg_sentiment_ratio'),
        'sentiment_positive':       safe_get(sent, 'positive_ratio'),
        'sentiment_hedging':        safe_get(sent, 'hedging_count'),
        'sentiment_assertive':      safe_get(sent, 'assertive_count'),
        'sentiment_hedge_rate':     safe_get(sent, 'hedge_rate'),
        'voice_pitch_mean':         safe_get(voice, 'pitch_mean'),
        'voice_pitch_std':          safe_get(voice, 'pitch_std'),
        'voice_pitch_range':        safe_get(voice, 'pitch_range'),
        'voice_voiced_fraction':    safe_get(voice, 'voiced_fraction'),
        'lexical_ttr':              safe_get(lex, 'type_token_ratio'),
        'lexical_mattr':            safe_get(lex, 'mattr'),
        'lexical_unique_words':     safe_get(lex, 'unique_words'),
        'lexical_rare_word_ratio':  safe_get(lex, 'rare_word_ratio'),
    }
    rows.append(row)

raw_df = pd.DataFrame(rows)
id_col = raw_df['response_id']
df = raw_df.drop(columns=['response_id']).apply(pd.to_numeric, errors='coerce')

print(f'DataFrame shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
display(df.head(3))

Loaded 86 successful records
DataFrame shape: (86, 29)
Columns (29): ['duration_s', 'fluency_wpm', 'fluency_pause_freq', 'fluency_mean_pause_dur', 'fluency_total_words', 'fluency_filler_count', 'fluency_filler_rate', 'fluency_scripted_score', 'intelligibility_conf', 'intelligibility_var_conf', 'pronounciation_score', 'discourse_connectors', 'discourse_tier1', 'discourse_tier2', 'sentiment_compound', 'sentiment_std', 'sentiment_neg_ratio', 'sentiment_positive', 'sentiment_hedging', 'sentiment_assertive', 'sentiment_hedge_rate', 'voice_pitch_mean', 'voice_pitch_std', 'voice_pitch_range', 'voice_voiced_fraction', 'lexical_ttr', 'lexical_mattr', 'lexical_unique_words', 'lexical_rare_word_ratio']


,duration_s,fluency_wpm,fluency_pause_freq,fluency_mean_pause_dur,fluency_total_words,fluency_filler_count,fluency_filler_rate,fluency_scripted_score,intelligibility_conf,intelligibility_var_conf,pronounciation_score,discourse_connectors,discourse_tier1,discourse_tier2,sentiment_compound,sentiment_std,sentiment_neg_ratio,sentiment_positive,sentiment_hedging,sentiment_assertive,sentiment_hedge_rate,voice_pitch_mean,voice_pitch_std,voice_pitch_range,voice_voiced_fraction,lexical_ttr,lexical_mattr,lexical_unique_words,lexical_rare_word_ratio
0,130.7100,113.8400,23,1.6000,248,11,0.0444,0,0.8875,0.0351,0.8291,5,0,16,0.1460,0.2622,0.0000,0.0870,1,2,0.4049,141.2900,64.8900,269.1000,0.4369,0.5721,0.7817,131,0.1119
1,1615.3500,23.7700,142,8.7500,640,56,0.0875,0,0.6123,0.0918,0.4203,9,1,79,0.7549,0.1791,0.0000,0.0000,2,3,0.3125,110.3900,41.9400,274.4500,0.0647,0.3830,0.6959,239,0.0522
2,1587.1200,44.9100,138,7.4700,1188,26,0.0219,0,0.6627,0.0994,0.5310,11,3,71,0.0505,0.2748,0.0132,0.0132,1,3,0.0843,189.1800,71.9300,275.0300,0.2075,0.3511,0.7610,401,0.1007


---
## Section 2 — Descriptive Statistics
For each feature: **mean, median, std, min, max, skewness, kurtosis, and the 10th/25th/50th/75th/90th percentiles**.  
This is the foundation for setting evaluation thresholds.

In [8]:
desc_stats = pd.DataFrame(index=df.columns)
desc_stats['mean']     = df.mean()
desc_stats['median']   = df.median()
desc_stats['std']      = df.std()
desc_stats['min']      = df.min()
desc_stats['p10']      = df.quantile(0.10)
desc_stats['p25']      = df.quantile(0.25)
desc_stats['p50']      = df.quantile(0.50)
desc_stats['p75']      = df.quantile(0.75)
desc_stats['p90']      = df.quantile(0.90)
desc_stats['max']      = df.max()
desc_stats['skewness'] = df.apply(lambda c: skew(c.dropna()))
desc_stats['kurtosis'] = df.apply(lambda c: kurtosis(c.dropna()))
desc_stats['n_valid']  = df.count()

display(desc_stats.round(4))

,mean,median,std,min,p10,p25,p50,p75,p90,max,skewness,kurtosis,n_valid
duration_s,1558.4322,1570.8150,491.3784,130.7100,900.8850,1296.3000,1570.8150,1879.5975,2161.0650,2608.5600,-0.2704,0.0720,86
fluency_wpm,70.5522,61.0850,38.2026,15.7200,30.3850,48.0900,61.0850,84.6500,111.8750,190.5000,1.3967,2.0233,86
fluency_pause_freq,146.8140,139.0000,65.5388,23.0000,67.5000,93.2500,139.0000,198.7500,229.5000,317.0000,0.4264,-0.5132,86
fluency_mean_pause_dur,6.1635,4.9900,4.3316,0.4100,2.3450,3.2000,4.9900,7.5875,11.0600,22.7400,1.4584,2.4601,86
fluency_total_words,1731.1744,1564.5000,963.7039,248.0000,763.5000,1143.0000,1564.5000,2047.7500,2847.0000,5777.0000,1.5029,3.2331,86
fluency_filler_count,66.7326,51.5000,59.7864,4.0000,12.5000,27.5000,51.5000,88.7500,134.0000,383.0000,2.3879,8.4711,86
fluency_filler_rate,0.0398,0.0335,0.0323,0.0053,0.0112,0.0208,0.0335,0.0532,0.0686,0.2557,3.6472,21.2950,86
fluency_scripted_score,0.9651,0.0000,2.9284,0.0000,0.0000,0.0000,0.0000,0.0000,3.5000,16.0000,3.4926,12.2394,86
intelligibility_conf,0.7405,0.7426,0.0791,0.4915,0.6543,0.6960,0.7426,0.7886,0.8346,0.9067,-0.3208,0.6538,86
intelligibility_var_conf,0.0820,0.0857,0.0199,0.0291,0.0536,0.0728,0.0857,0.0956,0.1014,0.1236,-0.6653,0.2645,86


---
## Section 3 — Bell Curves: Distribution Fitting per Feature
For each feature, plot the **empirical histogram** with a **fitted Normal distribution overlay** (bell curve).  
Also run the **Shapiro-Wilk normality test** and annotate the plot with W-statistic and p-value.  
Features that are NOT normal will need non-parametric thresholds (percentile-based) instead of z-score-based ones.

In [9]:
plt.style.use('dark_background')

numeric_cols = list(df.columns)
n_cols_per_row = 5
n_features = len(numeric_cols)
n_rows_per_fig = 6
n_per_fig = n_cols_per_row * n_rows_per_fig
n_figs = math.ceil(n_features / n_per_fig)

shapiro_results = []

for fig_idx in range(n_figs):
    chunk = numeric_cols[fig_idx * n_per_fig : (fig_idx + 1) * n_per_fig]
    n_rows = math.ceil(len(chunk) / n_cols_per_row)
    fig, axes = plt.subplots(n_rows, n_cols_per_row,
                             figsize=(n_cols_per_row * 4, n_rows * 3.2),
                             facecolor='#0d0d0d')
    axes = np.array(axes).flatten()

    for i, col in enumerate(chunk):
        ax = axes[i]
        ax.set_facecolor('#1a1a1a')
        data = df[col].dropna()

        # Histogram
        sns.histplot(data, ax=ax, color='#5dade2', alpha=0.7, kde=False, bins=20)

        # Fitted normal
        mu, std = norm.fit(data)
        x = np.linspace(data.min(), data.max(), 200)
        p = norm.pdf(x, mu, std)
        # Scale PDF to histogram count scale
        bin_width = (data.max() - data.min()) / 20 if data.max() != data.min() else 1
        ax2 = ax.twinx()
        ax2.set_facecolor('#1a1a1a')
        ax2.plot(x, p, color='#ff6b6b', linewidth=2.5, linestyle='--', label='Normal fit')
        ax2.set_yticks([])

        # Mean and median lines
        ax.axvline(data.mean(), color='white', linestyle='--', linewidth=1.2, alpha=0.8, label='Mean')
        ax.axvline(data.median(), color='yellow', linestyle=':', linewidth=1.2, alpha=0.8, label='Median')

        # Shapiro-Wilk
        W, p_val = shapiro(data)
        shapiro_results.append({'feature': col, 'W': W, 'p': p_val, 'is_normal': p_val > 0.05})

        color = '#00ff88' if p_val > 0.05 else '#ff4444'
        ax.set_title(col, fontsize=8, color='white', pad=3)
        ax.annotate(f'W={W:.3f}\np={p_val:.3f}', xy=(0.97, 0.95),
                    xycoords='axes fraction', ha='right', va='top',
                    fontsize=6.5, color=color,
                    bbox=dict(boxstyle='round,pad=0.2', fc='#0d0d0d', alpha=0.7))
        ax.tick_params(labelsize=6, colors='white')
        for spine in ax.spines.values():
            spine.set_edgecolor('#444')

    # Hide unused axes
    for j in range(len(chunk), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'Bell Curve Fits — Part {fig_idx + 1}', fontsize=14, color='white', y=1.01)
    fig.tight_layout()
    fname = OUT_DIR / f'bell_curves_part{fig_idx + 1}.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
    plt.close(fig)
    print(f'Saved: {fname}')

# Also save a combined bell_curves.png (copy of part 1)
import shutil
shutil.copy(OUT_DIR / 'bell_curves_part1.png', OUT_DIR / 'bell_curves.png')

shapiro_df = pd.DataFrame(shapiro_results)
print('\n--- Shapiro-Wilk Normality Test Results ---')
display(shapiro_df.round(4))
n_normal = shapiro_df['is_normal'].sum()
print(f'\nNormal (p>0.05): {n_normal}/{len(shapiro_df)} features')

Saved: /Users/samarthsrivastava/Audio Analysis/analysis_output/bell_curves_part1.png

--- Shapiro-Wilk Normality Test Results ---


,feature,W,p,is_normal
0,duration_s,0.9885,0.6511,True
1,fluency_wpm,0.8756,0.0000,False
2,fluency_pause_freq,0.9701,0.0434,False
3,fluency_mean_pause_dur,0.8802,0.0000,False
4,fluency_total_words,0.8896,0.0000,False
5,fluency_filler_count,0.7930,0.0000,False
6,fluency_filler_rate,0.7123,0.0000,False
7,fluency_scripted_score,0.3823,0.0000,False
8,intelligibility_conf,0.9819,0.2695,True
9,intelligibility_var_conf,0.9586,0.0076,False



Normal (p>0.05): 10/29 features


---
## Section 4 — Z-Score Normalization
Compute z-scores for all features. Z-score = (x - mean) / std.  
This converts every feature onto a common scale.  
Z > 2 = unusually high, Z < -2 = unusually low.  
This forms the basis of a **standardized scoring system**.

In [10]:
z_df = (df - df.mean()) / df.std()

# --- Violin plot ---
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(22, 7), facecolor='#0d0d0d')
ax.set_facecolor('#1a1a1a')

z_melted = z_df.melt(var_name='Feature', value_name='Z-score')
palette = sns.color_palette('plasma', n_colors=len(z_df.columns))
col_palette = {col: palette[i] for i, col in enumerate(z_df.columns)}

sns.violinplot(data=z_melted, x='Feature', y='Z-score', ax=ax,
               palette=col_palette, inner='quartile', linewidth=0.8, scale='width')

for y_val, color, ls in [(0, 'white', '--'), (2, '#ff6b6b', ':'), (-2, '#ff6b6b', ':')]:
    ax.axhline(y_val, color=color, linestyle=ls, linewidth=1.2, alpha=0.8)

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7.5, color='white')
ax.set_title('Z-Score Distributions — All Features (Violin)', fontsize=14, color='white')
ax.set_ylabel('Z-score', color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#444')

fig.tight_layout()
fig.savefig(OUT_DIR / 'zscore_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print('Saved: zscore_distributions.png')

# --- Box plot ---
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(22, 7), facecolor='#0d0d0d')
ax.set_facecolor('#1a1a1a')

sns.boxplot(data=z_melted, x='Feature', y='Z-score', ax=ax,
            palette=col_palette, flierprops=dict(marker='o', markerfacecolor='#ff6b6b', markersize=3, alpha=0.6))

for y_val, color, ls in [(0, 'white', '--'), (2, '#ff6b6b', ':'), (-2, '#ff6b6b', ':')]:
    ax.axhline(y_val, color=color, linestyle=ls, linewidth=1.2, alpha=0.8)

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7.5, color='white')
ax.set_title('Z-Score Distributions — All Features (Box)', fontsize=14, color='white')
ax.set_ylabel('Z-score', color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#444')

fig.tight_layout()
fig.savefig(OUT_DIR / 'zscore_boxplots.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print('Saved: zscore_boxplots.png')

# --- Outlier counts per feature ---
print('\n--- Z-score Outliers (|z| > 2) per Feature ---')
outlier_counts = (z_df.abs() > 2).sum()
outlier_df = outlier_counts.reset_index()
outlier_df.columns = ['feature', 'n_outliers_z2']
display(outlier_df)

Saved: zscore_distributions.png
Saved: zscore_boxplots.png

--- Z-score Outliers (|z| > 2) per Feature ---


,feature,n_outliers_z2
0,duration_s,3
1,fluency_wpm,6
2,fluency_pause_freq,3
3,fluency_mean_pause_dur,5
4,fluency_total_words,5
5,fluency_filler_count,3
6,fluency_filler_rate,2
7,fluency_scripted_score,6
8,intelligibility_conf,3
9,intelligibility_var_conf,7


---
## Section 5 — Percentile Bands: Evaluation Thresholds
Using empirical percentiles to define 5 performance bands per feature:

| Band | Percentile | Meaning |
|------|------------|---------|
| Poor | < 20th | Bottom 20% of all candidates |
| Below Average | 20th–40th | Below median |
| Average | 40th–60th | Around the median |
| Good | 60th–80th | Above median |
| Excellent | > 80th | Top 20% of all candidates |

These are the **data-driven thresholds** to use in your scoring rubric.

In [11]:
band_df = pd.DataFrame(index=df.columns)
band_df['p10']              = df.quantile(0.10)
band_df['poor_max_p20']     = df.quantile(0.20)
band_df['below_avg_max_p40']= df.quantile(0.40)
band_df['avg_max_p60']      = df.quantile(0.60)
band_df['good_max_p80']     = df.quantile(0.80)
band_df['p90']              = df.quantile(0.90)
band_df['excellent_above']  = df.quantile(0.80)   # same as good_max, excellent is > p80

print('=== Percentile Band Thresholds ===')
display(band_df.round(4))

# --- Horizontal stacked bar chart (normalized) ---
plt.style.use('dark_background')

feat_min = df.min()
feat_max = df.max()
feat_range = (feat_max - feat_min).replace(0, 1)  # avoid div by zero

p20_n = (band_df['poor_max_p20'] - feat_min) / feat_range
p40_n = (band_df['below_avg_max_p40'] - band_df['poor_max_p20']) / feat_range
p60_n = (band_df['avg_max_p60'] - band_df['below_avg_max_p40']) / feat_range
p80_n = (band_df['good_max_p80'] - band_df['avg_max_p60']) / feat_range
p100_n = (feat_max - band_df['good_max_p80']) / feat_range

features = list(df.columns)
band_colors = ['#c0392b', '#e67e22', '#f1c40f', '#27ae60', '#2980b9']
band_labels = ['Poor (<p20)', 'Below Avg (p20-40)', 'Average (p40-60)', 'Good (p60-80)', 'Excellent (>p80)']

fig, ax = plt.subplots(figsize=(14, max(8, len(features) * 0.42)), facecolor='#0d0d0d')
ax.set_facecolor('#1a1a1a')

left = np.zeros(len(features))
for seg, color, label in zip([p20_n, p40_n, p60_n, p80_n, p100_n], band_colors, band_labels):
    vals = seg.fillna(0).values
    ax.barh(features, vals, left=left, color=color, label=label, alpha=0.85, height=0.7)
    left += vals

ax.set_xlim(0, 1)
ax.set_xlabel('Normalized Feature Range (0=min, 1=max)', color='white', fontsize=10)
ax.set_title('Percentile Band Ranges — Normalized per Feature', fontsize=13, color='white')
ax.tick_params(colors='white', labelsize=8)
ax.legend(loc='lower right', fontsize=8, framealpha=0.5)
for spine in ax.spines.values():
    spine.set_edgecolor('#444')

fig.tight_layout()
fig.savefig(OUT_DIR / 'percentile_bands.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print('\nSaved: percentile_bands.png')

=== Percentile Band Thresholds ===


,p10,poor_max_p20,below_avg_max_p40,avg_max_p60,good_max_p80,p90,excellent_above
duration_s,900.8850,1211.0100,1434.6300,1675.3800,1919.2500,2161.0650,1919.2500
fluency_wpm,30.3850,44.6600,54.9500,67.2800,93.3200,111.8750,93.3200
fluency_pause_freq,67.5000,86.0000,116.0000,164.0000,205.0000,229.5000,205.0000
fluency_mean_pause_dur,2.3450,2.8200,4.3800,5.9300,8.7500,11.0600,8.7500
fluency_total_words,763.5000,956.0000,1449.0000,1769.0000,2126.0000,2847.0000,2126.0000
fluency_filler_count,12.5000,21.0000,42.0000,61.0000,97.0000,134.0000,97.0000
fluency_filler_rate,0.0112,0.0179,0.0250,0.0431,0.0579,0.0686,0.0579
fluency_scripted_score,0.0000,0.0000,0.0000,0.0000,0.0000,3.5000,0.0000
intelligibility_conf,0.6543,0.6794,0.7223,0.7590,0.7993,0.8346,0.7993
intelligibility_var_conf,0.0536,0.0687,0.0791,0.0893,0.0974,0.1014,0.0974



Saved: percentile_bands.png


---
## Section 6 — Feature Correlation Heatmap
Pearson correlation between all features.  
Highly correlated features (|r| > 0.7) may be **redundant** for scoring — you can drop one of them.  
Low-correlated features are **orthogonal dimensions** and should each be scored independently.

In [12]:
corr = df.corr(method='pearson')

# Mask upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

plt.style.use('dark_background')
n = len(corr)
fig, ax = plt.subplots(figsize=(max(14, n * 0.55), max(12, n * 0.55)), facecolor='#0d0d0d')
ax.set_facecolor('#1a1a1a')

sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    linewidths=0.3, linecolor='#333',
    cbar_kws={'shrink': 0.7}
)

ax.set_title('Pearson Correlation Heatmap — All Features', fontsize=14, color='white', pad=12)
ax.tick_params(colors='white', labelsize=8)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.setp(ax.get_yticklabels(), rotation=0)

fig.tight_layout()
fig.savefig(OUT_DIR / 'feature_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print('Saved: feature_correlation_heatmap.png')

# High-correlation pairs
print('\n--- Highly Correlated Feature Pairs (|r| > 0.65) ---')
pairs = []
cols = list(corr.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.65:
            pairs.append({'feature_a': cols[i], 'feature_b': cols[j], 'correlation': round(r, 4)})

if pairs:
    pairs_df = pd.DataFrame(pairs).sort_values('correlation', key=abs, ascending=False)
    display(pairs_df)
else:
    print('No pairs with |r| > 0.65 found.')

Saved: feature_correlation_heatmap.png

--- Highly Correlated Feature Pairs (|r| > 0.65) ---


,feature_a,feature_b,correlation
15,intelligibility_conf,pronounciation_score,0.9787
5,fluency_wpm,voice_voiced_fraction,0.9592
9,fluency_total_words,discourse_tier2,0.9514
14,intelligibility_conf,intelligibility_var_conf,-0.9079
16,intelligibility_var_conf,pronounciation_score,-0.8995
12,fluency_total_words,lexical_unique_words,0.8916
20,discourse_connectors,lexical_unique_words,0.8365
24,sentiment_hedging,sentiment_hedge_rate,0.8244
23,discourse_tier2,lexical_unique_words,0.7993
11,fluency_total_words,lexical_ttr,-0.7980


---
## Section 7 — Per-Dimension Summary for Evaluation Design
For each scoring dimension (Fluency, Intelligibility, Lexical, Discourse, Sentiment, Voice, Language Control),  
show which features belong to it and the recommended threshold values.

In [13]:
DIMENSIONS = {
    'FLUENCY': {
        'features': ['fluency_wpm', 'fluency_filler_rate', 'fluency_pause_freq', 'fluency_mean_pause_dur',
                     'fluency_total_words', 'fluency_filler_count', 'fluency_scripted_score'],
        'primary': 'fluency_wpm'
    },
    'INTELLIGIBILITY': {
        'features': ['intelligibility_conf', 'intelligibility_var_conf', 'pronounciation_score'],
        'primary': 'intelligibility_conf'
    },
    'LEXICAL RESOURCE': {
        'features': ['lexical_ttr', 'lexical_mattr', 'lexical_unique_words', 'lexical_rare_word_ratio'],
        'primary': 'lexical_mattr'
    },
    'DISCOURSE': {
        'features': ['discourse_connectors', 'discourse_tier1', 'discourse_tier2'],
        'primary': 'discourse_connectors'
    },
    'SENTIMENT': {
        'features': ['sentiment_compound', 'sentiment_std', 'sentiment_neg_ratio',
                     'sentiment_positive', 'sentiment_hedging', 'sentiment_assertive', 'sentiment_hedge_rate'],
        'primary': 'sentiment_compound'
    },
    'VOICE MODULATION': {
        'features': ['voice_pitch_mean', 'voice_pitch_std', 'voice_pitch_range', 'voice_voiced_fraction'],
        'primary': 'voice_voiced_fraction'
    },
    'LANGUAGE CONTROL': {
        'features': ['duration_s'],
        'primary': 'duration_s'
    }
}

def band_label(feature, p20, p40, p60, p80, val):
    if val < p20:   return 'Poor'
    elif val < p40: return 'Below Average'
    elif val < p60: return 'Average'
    elif val < p80: return 'Good'
    else:           return 'Excellent'

for dim_name, dim_info in DIMENSIONS.items():
    valid_features = [f for f in dim_info['features'] if f in df.columns]
    print(f"\n{'='*60}")
    print(f"  === {dim_name} DIMENSION ===")
    print(f"  Key features: {', '.join(valid_features)}")
    print(f"{'='*60}")

    for feat in valid_features:
        col = df[feat].dropna()
        if len(col) < 5:
            print(f'  {feat}: insufficient data\n')
            continue

        mn  = col.mean()
        med = col.median()
        sd  = col.std()
        p20 = col.quantile(0.20)
        p40 = col.quantile(0.40)
        p60 = col.quantile(0.60)
        p80 = col.quantile(0.80)
        sk  = skew(col)
        _, sw_p = shapiro(col) if len(col) >= 3 else (np.nan, np.nan)
        normality = f'normal fit p={sw_p:.3f}' if not np.isnan(sw_p) else 'n/a'
        skew_label = ('right-skewed' if sk > 0.5 else 'left-skewed' if sk < -0.5 else 'approx. symmetric')

        print(f'\n  {feat.upper()}:')
        print(f'    Mean: {mn:.4f} | Median: {med:.4f} | Std: {sd:.4f}')
        print(f'    Poor (<p20):        < {p20:.4f}')
        print(f'    Below Avg (p20-40): {p20:.4f} – {p40:.4f}')
        print(f'    Average   (p40-60): {p40:.4f} – {p60:.4f}')
        print(f'    Good      (p60-80): {p60:.4f} – {p80:.4f}')
        print(f'    Excellent (>p80):   > {p80:.4f}')
        print(f'    Distribution: {skew_label} (skewness={sk:.2f}), {normality}')

# --- KDE subplot figure (7 dimensions, primary feature each) ---
plt.style.use('dark_background')
n_dims = len(DIMENSIONS)
fig, axes = plt.subplots(1, n_dims, figsize=(n_dims * 4, 5), facecolor='#0d0d0d')

band_line_colors = ['#c0392b', '#e67e22', '#f1c40f', '#27ae60']
band_line_labels = ['p20', 'p40', 'p60', 'p80']

for idx, (dim_name, dim_info) in enumerate(DIMENSIONS.items()):
    ax = axes[idx]
    ax.set_facecolor('#1a1a1a')
    primary = dim_info['primary']

    if primary not in df.columns:
        ax.set_visible(False)
        continue

    col = df[primary].dropna()
    sns.kdeplot(col, ax=ax, color='#5dade2', fill=True, alpha=0.35, linewidth=2)

    for pct, color, lbl in zip([0.20, 0.40, 0.60, 0.80], band_line_colors, band_line_labels):
        thresh = col.quantile(pct)
        ax.axvline(thresh, color=color, linestyle='--', linewidth=1.5, label=f'{lbl}={thresh:.2f}')

    ax.set_title(f'{dim_name}\n({primary})', fontsize=8, color='white')
    ax.tick_params(colors='white', labelsize=6)
    ax.legend(fontsize=5.5, loc='upper right', framealpha=0.4)
    ax.set_xlabel('', color='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')

fig.suptitle('KDE + Band Thresholds — Primary Feature per Dimension', fontsize=12, color='white', y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / 'dimension_kde_thresholds.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print('\nSaved: dimension_kde_thresholds.png')


  === FLUENCY DIMENSION ===
  Key features: fluency_wpm, fluency_filler_rate, fluency_pause_freq, fluency_mean_pause_dur, fluency_total_words, fluency_filler_count, fluency_scripted_score

  FLUENCY_WPM:
    Mean: 70.5522 | Median: 61.0850 | Std: 38.2026
    Poor (<p20):        < 44.6600
    Below Avg (p20-40): 44.6600 – 54.9500
    Average   (p40-60): 54.9500 – 67.2800
    Good      (p60-80): 67.2800 – 93.3200
    Excellent (>p80):   > 93.3200
    Distribution: right-skewed (skewness=1.40), normal fit p=0.000

  FLUENCY_FILLER_RATE:
    Mean: 0.0398 | Median: 0.0335 | Std: 0.0323
    Poor (<p20):        < 0.0179
    Below Avg (p20-40): 0.0179 – 0.0250
    Average   (p40-60): 0.0250 – 0.0431
    Good      (p60-80): 0.0431 – 0.0579
    Excellent (>p80):   > 0.0579
    Distribution: right-skewed (skewness=3.65), normal fit p=0.000

  FLUENCY_PAUSE_FREQ:
    Mean: 146.8140 | Median: 139.0000 | Std: 65.5388
    Poor (<p20):        < 86.0000
    Below Avg (p20-40): 86.0000 – 116.0000
    A

---
## Section 8 — Outlier Analysis

In [14]:
# IQR-based outlier flags
Q1  = df.quantile(0.25)
Q3  = df.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

iqr_flag  = ((df < lower) | (df > upper)).astype(int)
z_extreme = (z_df.abs() > 3).astype(int)
combined_flag = ((iqr_flag + z_extreme) > 0).astype(int)

# Record-level summary
combined_flag_with_id = combined_flag.copy()
combined_flag_with_id.insert(0, 'response_id', id_col.values)
flag_summary = combined_flag.copy()
flag_summary['total_flags'] = combined_flag.sum(axis=1)
flag_summary['response_id'] = id_col.values
flagged_records = flag_summary[flag_summary['total_flags'] > 0].sort_values('total_flags', ascending=False)

flagged_cols = ['response_id', 'total_flags'] + [c for c in df.columns if combined_flag[c].sum() > 0]
print(f'Records with at least 1 outlier flag: {len(flagged_records)}')
display(flagged_records[flagged_cols].head(20))

# --- Outlier heatmap ---
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(max(16, len(df.columns) * 0.55), max(10, len(df) * 0.13)), facecolor='#0d0d0d')
ax.set_facecolor('#1a1a1a')

cmap = sns.color_palette(['#1a1a1a', '#c0392b'], as_cmap=True)
sns.heatmap(
    combined_flag, ax=ax,
    cmap=['#1a1a1a', '#c0392b'],
    cbar=True, cbar_kws={'label': 'Outlier (IQR or |z|>3)', 'shrink': 0.6},
    linewidths=0.05, linecolor='#2a2a2a',
    xticklabels=True, yticklabels=False
)
ax.set_title('Outlier Flag Heatmap (Red = Outlier)', fontsize=13, color='white', pad=10)
ax.set_xlabel('Feature', color='white', fontsize=9)
ax.set_ylabel('Candidate Record', color='white', fontsize=9)
ax.tick_params(colors='white', labelsize=7.5)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

fig.tight_layout()
fig.savefig(OUT_DIR / 'outlier_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print('Saved: outlier_heatmap.png')

print('\n--- Outlier Count per Feature (IQR + |z|>3 combined) ---')
outlier_per_feat = combined_flag.sum().reset_index()
outlier_per_feat.columns = ['feature', 'n_outliers']
display(outlier_per_feat.sort_values('n_outliers', ascending=False))

Records with at least 1 outlier flag: 43


,response_id,total_flags,duration_s,fluency_wpm,fluency_mean_pause_dur,fluency_total_words,fluency_filler_count,fluency_filler_rate,fluency_scripted_score,intelligibility_conf,intelligibility_var_conf,pronounciation_score,discourse_connectors,discourse_tier2,sentiment_compound,sentiment_std,sentiment_neg_ratio,sentiment_positive,sentiment_hedging,sentiment_assertive,sentiment_hedge_rate,voice_pitch_range,voice_voiced_fraction,lexical_ttr,lexical_mattr
16,69f3b509d22515a5798d3896,8,0,1,0,1,1,0,0,0,1,0,0,1,0,0,0,0,1,1,0,0,1,0,0
63,69ff13f47936a7a136a10b69,8,0,1,0,1,1,0,1,0,0,0,0,1,0,0,0,0,1,0,1,0,1,0,0
12,69f4461958f775c0f2adac62,7,0,1,0,1,0,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0,1,0,0
71,69fe24a162bc71d9b548ceb8,6,0,1,0,1,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0
0,69fee33a927f5c12e6d7eca3,5,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,1,0
24,69f36094541a84b24926afb9,5,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0
76,6a06e78239587121ed01bb0f,4,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0
72,6a06e8fa92b6101c22780d65,4,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0
84,69fedc3d70c221e8cd438094,3,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
18,6a02c900a865144394247c3e,3,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


Saved: outlier_heatmap.png

--- Outlier Count per Feature (IQR + |z|>3 combined) ---


,feature,n_outliers
7,fluency_scripted_score,13
18,sentiment_hedging,9
23,voice_pitch_range,8
19,sentiment_assertive,7
24,voice_voiced_fraction,7
1,fluency_wpm,6
3,fluency_mean_pause_dur,6
4,fluency_total_words,6
13,discourse_tier2,5
9,intelligibility_var_conf,5


---
## Summary
All statistical outputs saved.

In [15]:
print('=== Output Files ===')
png_files = sorted(OUT_DIR.glob('*.png'))
if not png_files:
    print('No PNG files found in output directory.')
else:
    for f in png_files:
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name:<45}  {size_kb:7.1f} KB')
print(f'\nTotal: {len(png_files)} files in {OUT_DIR}')

=== Output Files ===
  bell_curves.png                                  637.6 KB
  bell_curves_part1.png                            637.6 KB
  dimension_kde_thresholds.png                     214.7 KB
  feature_correlation_heatmap.png                  490.1 KB
  outlier_heatmap.png                              127.3 KB
  percentile_bands.png                             127.9 KB
  zscore_boxplots.png                              142.6 KB
  zscore_distributions.png                         288.0 KB

Total: 8 files in /Users/samarthsrivastava/Audio Analysis/analysis_output
